# `PointNet Segmentation`


#### Фамилия, имя:

Дата выдачи: <span style="color:red">__24 февраля__</span>.

Дедлайн: <span style="color:red">__10 марта 23:30__</span>.

Стоимость: __10 баллов__

<span style="color:red">__В ноутбуке все клетки должны выполняться без ошибок при последовательном их выполнении.__</span>

### Getting started



#### **Цель работы:**

В этом ноутбуке мы обучим PointNet на задачу **part segmentation** на датасете **ShapeNetPart**:
для каждого 3D-объекта (например, Chair) нужно предсказать **метку части** для каждой точки (например, сиденье/спинка/ножки).

In [ ]:
import os
import json
import math
import random
import shutil
import subprocess
from pathlib import Path
from tqdm.autonotebook import tqdm

import torch
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

/tmp/ipython-input-2105807992.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

### Data

Для датасета ShapeNetPart:

- Есть **16 категорий объектов** (Airplane, Chair, Table, …).
- Для каждой категории определён свой набор **part-меток** (всего 50 уникальных меток на весь датасет).
- В каждом примере (.txt файл): облако точек (xyz + нормали) и разметка part-label для каждой точки.

В этом ноутбуке мы используем только **xyz** (без нормалей), чтобы модель была минималистичной.

In [ ]:
CATEGORY_IDS = {
    "Airplane":   "02691156",
    "Bag":        "02773838",
    "Cap":        "02954340",
    "Car":        "02958343",
    "Chair":      "03001627",
    "Earphone":   "03261776",
    "Guitar":     "03467517",
    "Knife":      "03624134",
    "Lamp":       "03636649",
    "Laptop":     "03642806",
    "Motorbike":  "03790512",
    "Mug":        "03797390",
    "Pistol":     "03948459",
    "Rocket":     "04099429",
    "Skateboard": "04225987",
    "Table":      "04379243",
}


# Which part-labels (0..49) are valid for each object category:
SEG_CLASSES = {
    "Airplane":   [0, 1, 2, 3],
    "Bag":        [4, 5],
    "Cap":        [6, 7],
    "Car":        [8, 9, 10, 11],
    "Chair":      [12, 13, 14, 15],
    "Earphone":   [16, 17, 18],
    "Guitar":     [19, 20, 21],
    "Knife":      [22, 23],
    "Lamp":       [24, 25, 26, 27],
    "Laptop":     [28, 29],
    "Motorbike":  [30, 31, 32, 33, 34, 35],
    "Mug":        [36, 37],
    "Pistol":     [38, 39, 40],
    "Rocket":     [41, 42, 43],
    "Skateboard": [44, 45, 46],
    "Table":      [47, 48, 49],
}

CAT_NAMES = list(CATEGORY_IDS.keys())
SYNSET_TO_CATNAME = {v: k for k, v in CATEGORY_IDS.items()}
SYNSET_TO_IDX = {CATEGORY_IDS[name]: i for i, name in enumerate(CAT_NAMES)}
IDX_TO_CATNAME = {i: name for i, name in enumerate(CAT_NAMES)}

NUM_OBJ_CLASSES = 16
NUM_PART_LABELS = 50

In [ ]:
from pathlib import Path
import shutil
import urllib.request

ZIP_NAME = "shapenetcore_partanno_segmentation_benchmark_v0_normal.zip"
DIR_NAME = "shapenetcore_partanno_segmentation_benchmark_v0_normal"
URL = ("https://huggingface.co/datasets/wangps/shapenet_segmentation/resolve/main/"
       "shapenetcore_partanno_segmentation_benchmark_v0_normal.zip")

def download_shapenetpart(root=""):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)

    data_dir = root / DIR_NAME
    if (data_dir / "train_test_split").exists():
        return data_dir

    zip_path = root / ZIP_NAME
    if not zip_path.exists():
        urllib.request.urlretrieve(URL, zip_path)

    shutil.unpack_archive(zip_path, root)
    return data_dir

### Dataset

Реализуем в начале какое-то количество самых базовых аугументаций.

**Важно:** в задании вы можете добавить свои варианты аугументаций для улучшения качества и устойчивости модели, но обязательно напишите об этом!

In [ ]:
def normalize_pc(xyz: np.ndarray) -> np.ndarray:
    xyz = xyz - xyz.mean(axis=0, keepdims=True)
    scale = np.max(np.linalg.norm(xyz, axis=1))
    if scale > 0:
        xyz = xyz / scale
    return xyz

def rotate_z(xyz: np.ndarray) -> np.ndarray:
    theta = random.random() * 2.0 * math.pi
    R = np.array([[math.cos(theta), -math.sin(theta),   0],
                  [math.sin(theta), math.cos(theta),    0],
                  [0,               0,                  1]], dtype=np.float32)
    return xyz @ R.T

def add_noise(xyz: np.ndarray, sigma=0.02) -> np.ndarray:
    noise = np.random.normal(0, sigma, (xyz.shape)).astype(np.float32)
    return xyz + noise


Хотим, чтобы датасет по индексу возвращал следующее:
- `xyz`: (N, 3) координаты точек
- `cls`: класс объекта (0..15)
- `seg`: (N,) метка части для каждой точки

Важные моменты:
1) Берём случайное подмножество из N точек
2) Нормализуем облако (центрируем и масштабируем)
3) В `train` добавляем аугментации (поворот вокруг оси Z + шум)

In [ ]:
class ShapeNetPartTxt(Dataset):
    """
    Each shape is a .txt with columns:
      x y z nx ny nz segLabel
    We'll use only xyz for PointNet (minimal).
    """
    def __init__(self, root_dir, split="train", num_points=1024, eval_seed=0):
        assert split in ("train", "val", "test")
        self.root = Path(root_dir)
        self.split = split
        self.n = int(num_points)
        self.eval_seed = int(eval_seed)
        self.train = (split == "train")

        split_file = self.root / "train_test_split" / f"shuffled_{split}_file_list.json"
        raw = json.loads(split_file.read_text())

        allowed = set(CATEGORY_IDS.values())
        items = []
        for s in raw:
            parts = Path(s).parts
            synset = next((p for p in parts if p in allowed), None)
            if synset is None:
                continue
            shape = Path(parts[-1]).with_suffix(".txt").name  # "xxx.txt"
            p = self.root / synset / shape
            items.append((p, SYNSET_TO_IDX[synset]))

        self.items = items

    def __len__(self):
        return len(self.items)

    def _choice(self, P, idx):
        replace = P < self.n
        if self.train:
            return np.random.choice(P, self.n, replace=replace)
        rng = np.random.default_rng(self.eval_seed + idx)
        return rng.choice(P, self.n, replace=replace)

    def __getitem__(self, idx):
        path, cls_idx = self.items[idx]

        arr = np.loadtxt(path, dtype=np.float32)
        xyz = arr[:, :3]
        seg = arr[:, -1].astype(np.int64)

        choice = self._choice(len(xyz), idx)
        xyz, seg = xyz[choice], seg[choice]

        xyz = normalize_pc(xyz)
        if self.train:
            xyz = add_noise(rotate_z(xyz))

        return (
            torch.from_numpy(xyz),                 # (N, 3) float32
            torch.tensor(cls_idx, dtype=torch.long),
            torch.from_numpy(seg),                 # (N,) int64
        )

### PointNet (3 балла)

В этом блоке надо реализовать PointNet для задачи сегментации.

Подробнее про архитектуру вы можете посмотреть у нас на [семинаре](https://github.com/struminsky/hse_3dcv/blob/main/week_05/seminar.ipynb) или в оригинальной [статье](https://arxiv.org/abs/1612.00593).


`Важно:`   
Давайте считать, что в нашей постановке задачи считается, что **категория объекта известна** (например, это Chair), поэтому модель получает **cls_label** и добавляет one-hot в признаки каждой точки. Это стандартный режим для ShapeNetPart: разные категории имеют разные допустимые части,
и обуславливание помогает модели не путаться между разными частями разных объектов.

In [ ]:
class TNet(nn.Module):
    def __init__(self, k: int):
        super().__init__()
        self.k = k
        ???

    def forward(self, x):   # (B, k, N)
        ???

class PointNetEncoder(nn.Module):
    def __init__(self, feature_transform=True):
        super().__init__()
        ???

    def forward(self, x):   # (B, 3, N)
        ???


class PointNetPartSeg(nn.Module):
    """
    Point-wise part segmentation with category conditioning (one-hot class).
    """
    def __init__(self, num_obj_classes=16, num_parts=50, feature_transform=True):
        super().__init__()
        self.num_obj_classes = num_obj_classes
        self.num_parts = num_parts

        self.encoder = PointNetEncoder(feature_transform=feature_transform)

        ???

    def forward(self, x, cls_label):  # x: (B, 3, N), cls_label: (B,)
        ???

### Training (3 балла)

Оптимизируем `CrossEntropy` по меткам частей (segmentation label per point).

Дополнительно (если включён feature transform):
- добавляется регуляризатор ортогональности матрицы преобразования,
  чтобы она не “ломала” геометрию и не вырождалась.

Метрики:
- **Accuracy**: доля правильно размеченных точек
- **Instance mIoU**: mIoU по частям внутри каждого объекта, затем среднее по всем объектам
- **Class mIoU**: среднее Instance mIoU по категориям объектов (16 категорий)

In [1]:
def feature_transform_regularizer(trans):  # (B,k,k)
    """We want orthogonality to preserve distances and angles, which is critical for invariance to rotations/scaling."""
    B, k, _ = trans.size()
    I = torch.eye(k, device=trans.device).unsqueeze(0).repeat(B, 1, 1)
    diff = torch.bmm(trans, trans.transpose(1, 2)) - I
    return torch.mean(torch.norm(diff, dim=(1, 2)))

def compute_shape_iou(pred_b, seg_b, cat_name):
    """Instance mIoU (standard for ShapeNetPart)"""
    parts = SEG_CLASSES[cat_name]
    ious = []
    for part in parts:
        I = ((pred_b == part) & (seg_b == part)).sum().item()
        U = ((pred_b == part) | (seg_b == part)).sum().item()
        ious.append(1.0 if U == 0 else I / U)
    return float(np.mean(ious))


In [ ]:
def train_one_epoch(model, loader, optimizer, device, alpha=1e-3):
    model.train()
    total_loss, total_correct, total_points = 0.0, 0, 0

    for xyz, cls, seg in tqdm(loader, desc="Train", leave=False):
        ???

    return total_loss / max(len(loader), 1), total_correct / max(total_points, 1)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, total_correct, total_points = 0.0, 0, 0
    shape_ious = []
    per_class_ious = {name: [] for name in CAT_NAMES}

    for xyz, cls, seg in tqdm(loader, desc="Eval", leave=False):
        ???

    # inst-mIoU усредняет IoU по всем объектам в датсете, показывая качество сегментации каждой отдлеьной формы
    # cls-mIoU усредняет IoU по всем классам частей, показывая насколько хорошо модель различате редкие типы застей
    inst_miou = float(np.mean(shape_ious)) if shape_ious else 0.0
    cls_miou = float(np.mean([np.mean(v) for v in per_class_ious.values() if len(v) > 0])) if shape_ious else 0.0

    return (
        total_loss / max(len(loader), 1),
        total_correct / max(total_points, 1),
        inst_miou,
        cls_miou,
    )

Тут дальше начинается творчество в обучении. Можете крутить и вертеть как хотите гиперпараметры (**остальные менять нельзя!**), которые помечены знаками вопросов, и усложнять пайплайн всякими базовыми штуками из DL.

**Баллы распределяются по результатам на Test:**
* `1 балл из 3 баллов`: правильно реализованы функции `train_one_epoch()` и `evaluate()`
* `2 балла из 3 баллов` : inst-mIoU > 70% и cls-mIoU > 60%
* `3 балла из 3 баллов` : inst-mIoU > 75% и cls-mIoU > 65%

In [ ]:
seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_dir = download_shapenetpart()

num_points = 1024
batch_size = ???
epochs = ???
lr = ???

train_ds = ShapeNetPartTxt(data_dir, split="train", num_points=num_points)
val_ds   = ShapeNetPartTxt(data_dir, split="val",   num_points=num_points)
test_ds  = ShapeNetPartTxt(data_dir, split="test",  num_points=num_points)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)

model = PointNetPartSeg(num_obj_classes=NUM_OBJ_CLASSES, num_parts=NUM_PART_LABELS, feature_transform=True)
model.to(device)

optimizer = ???

best_val_miou = -1.0
for ep in tqdm(range(1, epochs + 1)):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, device, alpha=1e-3)
    va_loss, va_acc, va_inst_miou, va_cls_miou = evaluate(model, val_loader, device)

    print(
        f"Epoch {ep:02d}/{epochs} | "
        f"train loss {tr_loss:.4f} acc {tr_acc*100:.2f}% | "
        f"val loss {va_loss:.4f} acc {va_acc*100:.2f}% | "
        f"val inst-mIoU {va_inst_miou*100:.2f}% cls-mIoU {va_cls_miou*100:.2f}%"
    )

    if va_inst_miou > best_val_miou:
        best_val_miou = va_inst_miou
        torch.save(model.state_dict(), "pointnet_partseg_best.pth")

test_loss, test_acc, test_inst_miou, test_cls_miou = evaluate(model, test_loader, device)
print(
    f"TEST | loss {test_loss:.4f} acc {test_acc*100:.2f}% | "
    f"inst-mIoU {test_inst_miou*100:.2f}% cls-mIoU {test_cls_miou*100:.2f}%"
)

### Эксперименты после обучения (4 балла)

Ниже предложен набор экспериментов, которые надо провести **с уже обученной моделью**.

#### Влияние числа точек на качество (1 балл)

**Цель:** исследовать, как количество входных точек влияет на качество предсказания сегментации частей объектов (accuracy / inst-mIoU / cls-mIoU).

**Идеи:**
- Модель обучалась на 1024 точках, однако на инференсе число входных точек может вариироваться.
- Для каждого значения из заданного диапазона (например, 64, 128, 256, 512, 1024, 2048) замерьте три метрики: accuracy, Instance mIoU и Class mIoU.
- По результатам постройте график "доля точек - качество"
- Опишите результаты

**Вопросы:**
- Каково минимальное число точек, при котором модель сохраняет приемлемое качество?
- Какие метрики деградируют быстрее при сильном разрежении облака точек: accuracy или mIoU?

In [ ]:
#TODO

#### Устойчивость к поворотам (1 балл)

**Цель:** оценить, насколько модель устойчива к поворотам входного облака точек — как к контролируемым (вокруг вертикальной оси), так и к произвольным (вокруг случайной оси).

**Идеи:**
- Применять геометрическую трансформацию к координатам xyz на test/val перед подачей в модель, без переобучения все модели
- Сценарий (A): поворот вокруг оси Z на варьируемый угол
- Сценарий (B): поворот вокруг случайной оси на случайный угол
- Для каждого сценария замерить accuracy, Instance mIoU и Class mIoU и сравнить с базовой оценкой без поворота
- Опишите результаты


**Вопросы:**
- Является ли деградация монотонной с ростом угла поворота, или существует пороговое значение, после которого качество резко падает?
- Где просадка качества больше: при повороте вокруг оси Z или вокруг случайной оси?

In [ ]:
#TODO

#### Добавление шума в координаты (1 балл)

**Цель:** оценить устойчивость модели к шуму в координатах xyz и определить, при каком уровне зашумления качество сегментации становится неприемлемым.

**Идеи:**
- Выбрать уровни шума \sigma: [0.0, 0.005, 0.01, 0.02, 0.05, 0.1]
- Добавить случайный шум N(0, \sigma) к координатам после нормализации
- Замерить accuracy, Instance mIoU и Class mIoU для каждого уровня шума
- Построить график зависимости "\sigma - качество"
- Опишите результаты

**Вопросы:**
- Существует ли пороговое значение \sigma, после которого качество резко падает, или деградация происходит плавно?
- Какие категории объектов и их части наиболее чувствительны к зашумлению координат?

In [ ]:
#TODO

#### Анализ ошибок по категориям (1 балл)

**Цель:** выявить наиболее сложные для модели категории объектов и детально исследовать характер ошибок через визуализацию худших предсказаний.

**Идеи:**
- Посчитать Instance mIoU отдельно для каждой категории (Airplane, Chair, ...) и отсортировать от худших к лучшим
- Вывести таблицу «category - mean Instance mIoU» и выделить топ-3 худших и топ-3 лучших категорий
- Для худших категорий найти объекты с минимальным Instance mIoU и визуализировать их в 3D: цвет точек по ground truth и по предсказанию модели рядом
- Опишите результаты

**Вопросы:**
- Почему именно эти категории оказались сложнее — из-за геометрии, схожих между собой частей или малого числа обучающих примеров?
- Какие типы ошибок повторяются чаще всего, и есть ли конкретные пары частей, которые модель систематически путает?

In [ ]:
#TODO